# Fantasy Football Hierarchical Bayesian Inference

## Imports

In [1]:
%matplotlib inline
%config InlineBackend.figure_format = 'png'

import os
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
import pandas as pd
import pickle
from IPython.core.debugger import Tracer;

## Data Import and Munging

### nfldb
Create data base with metrics for the 2015 regular season with nfldb

In [ ]:
from ffbhm import *
from nfldbim import *

# model parameters
metric='score' # evience
K = 1 # evidence scaling
burnin = 3000 # burnin 
nsims=1000 # number of simulation to perform
nu=3. # prior
sd=2.5 # prior
season_year = 2017



In [ ]:
df_bets = pd.DataFrame()
# spread for the year (file generated from cbs_scraper.)
df_spread = pd.read_csv("../ruby_scrapers/spread.csv", sep=',', delimiter=None, header='infer')
nreg = 4
for season_year in range(2013, 2018):
    for week in range(5, 18):
        # load this year's game history
        df, teams = nfldbim(season_year, range(week-nreg, week+1))

        # this weeks games
        df_games = df[df.week == week].copy()

        # bayesian inference approximation
        fname = 'data_%d_%d_%d.pickle' % (season_year, week, nreg)
        # try loading data
        if os.path.isfile(fname):
            with open(fname, 'rb') as handle:
                trace, scores, hdis, simulations = pickle.load(handle)
        else:        
            # bayes hierarchical model
            trace = bhm(df, metric, K, nu, sd)

            # simulate this weeks games
            simulations = simulate_weeks(df_games, trace, nsims=nsims, metric=metric, K=K, burnin=burnin)

            # predict this weeks scores
            scores, hdis = predictions(df_games, simulations, teams, nsims=nsims, metric='score')

            # save data
            with open(fname, 'wb') as buff:
                pickle.dump((trace, scores, hdis, simulations), buff)

        num_games = float(df_games.shape[0])

        # vegas spread.
        df_spread_week = df_spread[(df_spread.week==week) & (df_spread.year==season_year)]

        # merge spread with weekly weekly data
        df_games = pd.merge(df_games,
                        df_spread_week[['home_team','home_spread']],
                        on='home_team')

        # split simulations by home or away
        week_record_home = scores[df_games.home_team]
        week_record_away = scores[df_games.away_team]

        # compute spread for each simulation
        week_record = week_record_home  - week_record_away.values
        week_spread = week_record[df_games.home_team] + df_games.home_spread.values

        # mean predicted spread
        df_games['pred_spread'] = week_spread.median().values

        # compute odds that the home team wins
        df_games['odds_home_win'] = (week_record[df_games.home_team]
                                       > 0).mean().values

        # compute odds that the home team wins against the spread
        df_games['odds_home_win_spread'] = (week_spread[df_games.home_team]
                                               > 0).mean().values

        # should we bet on home and what are the odds that we win the bet
        df_games['bet_on_home'] = df_games.apply(lambda x:
                                         x.odds_home_win > 0.5,
                                         axis=1)

        # odds that the home team will win
        df_games['odds_win'] = df_games.apply(lambda x:
                                      np.abs(x.odds_home_win - 0.5) + 0.5,
                                      axis=1)

        # set all nans to zero probability
        df_games.loc[np.isnan(df_games.home_spread),'odds_win'] = 0

        # find true spread from game outcome
        df_games['true_spread'] = df_games.apply(lambda x:
                                         x.away_score - x.home_score,
                                         axis=1)    

        # does home win against the spread?
        df_games['home_win_spread'] = df_games.apply(lambda x:
                                      x.home_score - x.away_score + x.home_spread > 0,
                                      axis=1)

        # would we have won this bet
        df_games['win_bet'] = df_games.bet_on_home == df_games.home_win_spread

        # what if I only take high probability bets
        df_bets = df_bets.append(df_games[(df_games.odds_win<1) & (df_games.odds_win>0.65)])
        print('Week All:%f, Week:%f, Total:%f' % (df_games.win_bet.mean(),df_bets[df_bets.week == week].win_bet.mean(), df_bets.win_bet.mean()) )

/Users/edgar/Library/Environments/ff/lib/python2.7/site-packages/pymc3/tuning/starting.py:91: UserWarning: In future versions, set the optimization algorithm with a string. For example, use `method="L-BFGS-B"` instead of `fmin=sp.optimize.fmin_l_bfgs_b"`.
  warnings.warn('In future versions, set the optimization algorithm with a string. '
logp = -551.32:  27%|██▋       | 13400/50000 [00:08<00:24, 1505.13it/s] 

Optimization terminated successfully.
         Current function value: 551.323684
         Iterations: 18
         Function evaluations: 13401


100%|██████████| 10500/10500 [00:39<00:00, 264.87it/s]
ffbhm.py:205: FutureWarning: by argument to sort_index is deprecated, please use .sort_values(by=...)
  hdis = hdis.sort_index(by=metric)


Week All:0.642857, Week:0.700000, Total:0.700000


logp = -555.35:  19%|█▉        | 9654/50000 [00:06<00:26, 1499.70it/s] 


...

100%|██████████| 10500/10500 [00:52<00:00, 200.85it/s]

Week All:0.625000, Week:0.800000, Total:0.741935


logp = -612.74:  24%|██▍       | 12001/50000 [00:08<00:26, 1423.18it/s]   


Optimization terminated successfully.
         Current function value: 612.735031
         Iterations: 16
         Function evaluations: 12001


 77%|███████▋  | 8049/10500 [00:35<00:10, 228.34it/s]